# 06 - Evaluation: Models Evaluation Comparison


In this notebook, we perform a comparative analysis of all trained models using:
- Metric distribution analysis and comparison (plots and summary statistics)
- Statistical significance testing (Corrected Resampled t-test)


## Import libraries and set the paths

In [1]:
from __future__ import annotations

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scikit_posthocs as sp
from pathlib import Path

from fraud_dynamic_ensemble.config import MODELS_DIR, FIGURES_DIR
from fraud_dynamic_ensemble.evaluation.statistical_test_evaluation import corrected_resampled_ttest

2026-01-24 17:10:11.514 | INFO     | fraud_dynamic_ensemble.config:<module>:14 - PROJ_ROOT path is: /home/leonardosaccotelli/Desktop/UNIVERSITA/MACHINE-LEARNING/Dynamic-Ensemble-Learning-for-Credit-Card-Fraud-Detection


In [2]:
EXPERIMENT_NAME = "CostSensitiveLearning___RandomizedSearchCV__niter_30__cv_5"

In [3]:
models_results_path: Path = MODELS_DIR / EXPERIMENT_NAME
print(f"Loading results at path:\n\t{models_results_path}")

Loading results at path:
	/home/leonardosaccotelli/Desktop/UNIVERSITA/MACHINE-LEARNING/Dynamic-Ensemble-Learning-for-Credit-Card-Fraud-Detection/models/CostSensitiveLearning___RandomizedSearchCV__niter_30__cv_5


In [4]:
FIGURES_MODELS_COMPARISON_DIR = FIGURES_DIR / "EV_models_comparison_evaluation"
FIGURES_MODELS_COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

FIGURES_MODELS_COMPARISON_EXPERIMENT_DIR = FIGURES_MODELS_COMPARISON_DIR / EXPERIMENT_NAME
FIGURES_MODELS_COMPARISON_EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

FIGURES_MULTIPLE_MODELS_EVALUATION_DIR = FIGURES_MODELS_COMPARISON_EXPERIMENT_DIR / "multiple_models_evaluation"
FIGURES_MULTIPLE_MODELS_EVALUATION_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
pd.set_option("display.max_columns", None)
plt.rcParams.update({"font.size": 16})
sns.set_style("whitegrid")
sns.set_palette("tab10")

## Data Loading and Basic Overview

In [6]:
files = list(models_results_path.glob("*/generalization_metrics_summary.csv"))
generalization_df = pd.DataFrame()

print(f"Found {len(files)} files. Loading...")

if files:
    # Read and Concatenate
    # We use a generator expression inside concat for memory efficiency
    generalization_df = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)

    print("Success! Combined dataframe shape:", generalization_df.shape)
else:
    print(f"No files found in {models_results_path.absolute()}")

Found 23 files. Loading...
Success! Combined dataframe shape: (2300, 25)


In [7]:
generalization_df

,experiment_name,iteration,fold,model,split,tp,tn,fp,fn,accuracy,precision,recall,f1,specificity,fpr,balanced_accuracy,geometric_mean,mcc,kappa,roc_auc,average_precision,score_time,fold_size,selected_features_indices,selected_features_names
0,CostSensitiveLearning___RandomizedSearchCV__ni...,1,1,KNOP,generalization,44,2415,45,3,0.980854,0.494382,0.936170,0.647059,0.981707,0.018293,0.958939,0.958668,0.672788,0.638181,0.971969,0.947270,9.851874,2507,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16...","['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8..."
1,CostSensitiveLearning___RandomizedSearchCV__ni...,1,2,KNOP,generalization,40,2424,36,7,0.982848,0.526316,0.851064,0.650407,0.985366,0.014634,0.918215,0.915756,0.661678,0.642115,0.942350,0.824992,6.699205,2507,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14,...","['Amount_log1p', 'V1', 'V2', 'V3', 'V4', 'V5',..."
2,CostSensitiveLearning___RandomizedSearchCV__ni...,1,3,KNOP,generalization,39,2435,25,8,0.986837,0.609375,0.829787,0.702703,0.989837,0.010163,0.909812,0.906286,0.704823,0.696133,0.969495,0.856795,10.615066,2507,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16...","['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8..."
3,CostSensitiveLearning___RandomizedSearchCV__ni...,1,4,KNOP,generalization,39,2439,21,8,0.988432,0.650000,0.829787,0.728972,0.991463,0.008537,0.910625,0.907030,0.728785,0.723151,0.940369,0.835085,8.374084,2507,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","['Amount_log1p', 'V1', 'V2', 'V3', 'V4', 'V5',..."
4,CostSensitiveLearning___RandomizedSearchCV__ni...,1,5,KNOP,generalization,41,2379,80,7,0.965297,0.338843,0.854167,0.485207,0.967466,0.032534,0.910817,0.909053,0.525351,0.470695,0.925054,0.850371,13.076344,2507,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16...","['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2295,CostSensitiveLearning___RandomizedSearchCV__ni...,10,6,APriori,generalization,42,2379,80,6,0.965696,0.344262,0.875000,0.494118,0.967466,0.032534,0.921233,0.920072,0.536572,0.479823,0.961341,0.840800,7.278489,2507,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14, 16...","['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8..."
2296,CostSensitiveLearning___RandomizedSearchCV__ni...,10,7,APriori,generalization,39,2394,65,9,0.970483,0.375000,0.812500,0.513158,0.973566,0.026434,0.893033,0.889395,0.540214,0.500059,0.940215,0.726804,7.918153,2507,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","['Amount_log1p', 'V1', 'V2', 'V3', 'V4', 'V5',..."
2297,CostSensitiveLearning___RandomizedSearchCV__ni...,10,8,APriori,generalization,40,2364,95,7,0.959298,0.296296,0.851064,0.439560,0.961366,0.038634,0.906215,0.904535,0.488182,0.423521,0.950283,0.804343,14.182767,2506,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 14,...","['Amount_log1p', 'V1', 'V2', 'V3', 'V4', 'V5',..."
2298,CostSensitiveLearning___RandomizedSearchCV__ni...,10,9,APriori,generalization,38,2420,39,9,0.980846,0.493506,0.808511,0.612903,0.984140,0.015860,0.896325,0.892013,0.623090,0.603672,0.912103,0.727781,10.022541,2506,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","['Amount_log1p', 'V1', 'V2', 'V3', 'V4', 'V5',..."


## Fix the metrics and general settings

In [8]:
# Define your metrics
metrics_to_analyze = [
    "balanced_accuracy",
    "mcc",
    "average_precision",
    "f1",
]
print(f"Selected metrics:\n\t{metrics_to_analyze}")

# Get list of unique models
unique_models = generalization_df["model"].unique()
print(f"Selected models:\n\t{unique_models}")

Selected metrics:
	['balanced_accuracy', 'mcc', 'average_precision', 'f1']
Selected models:
	['KNOP' 'KNORAE' 'APosteriori' 'KNeighborsClassifier' 'Exponential'
 'LogitBoostClassifier' 'BalancedRandomForestClassifier'
 'DecisionTreeClassifier' 'StackingClassifier' 'DESKL' 'XGBClassifier'
 'Logarithmic' 'RUSBoostClassifier' 'MLA' 'RandomForestClassifier'
 'KNORAU' 'MLPClassifier' 'ExtraTreesClassifier' 'VotingClassifier' 'RRC'
 'METADES' 'DESP' 'APriori']


## Model Comparison (Distribution Analysis)

### Objective
To compare the performance of all models on the unseen data (Generalization Split) across multiple metrics.

### Methodology
We generate **individual distribution plots** for each metric:
* **Scope:** Generalization split only.
* **Visualization:**
    * **Boxplot:** Displays the median (central line), Interquartile Range (box), and overall spread.
    * **Strip Plot:** Overlays the raw data points (100 folds) to reveal the density and detect outliers.
* **Sorting:** Models are sorted **Alphabetically by Name** on the X-axis. This fixed ordering allows for easy visual cross-referencing between different metric plots.
* **Scale:** All plots are fixed to a $0.0 - 1.0$ Y-axis range for consistent comparison.

### Interpretation Guide
* **Performance Level:** Compare the height of the boxes. Higher is better for standard metrics (Accuracy, F1, MCC).
* **Reliability Check:** Look at the vertical size of the box. A short box indicates a consistent model. A tall box indicates high variance (unstable performance).
* **Outliers:** Individual dots floating far below the box indicate specific partitions where the model failed.

In [9]:
def compare_models_distribution(df, metrics_list, save_path):
    """
    Compare model performance distributions on the generalization split using per-metric plots.

    This function filters the input results to the ``"generalization"`` split and generates
    one figure per metric in ``metrics_list`` to compare model score distributions. For each
    metric, it produces a boxplot (summary statistics, with boxplot fliers hidden) overlaid
    with a strip plot (raw per-fold/per-iteration points). Figures are saved individually to
    ``save_path``.

    Models are ordered alphabetically by the ``"model"`` column to ensure consistent x-axis
    ordering across different metric plots. In addition to the per-metric plots, the function
    writes a CSV summary table containing descriptive statistics (via
    ``groupby("model")[metrics_list].describe()``) and sorts model columns alphabetically for
    consistent reporting.

    Parameters
    ----------
    df : pandas.DataFrame
        Results DataFrame. Must contain:
        - ``"model"``: model identifier used for grouping and x-axis ordering,
        - ``"split"``: split identifier used to select ``"generalization"`` rows, and
        - one numeric column per entry in ``metrics_list``.
    metrics_list : Sequence[str]
        Metric column names to plot and summarize. Each metric must be present in ``df``.
        The plots assume metrics are in ``[0, 1]`` (y-axis is fixed to ``[0, 1.05]``).
    save_path : pathlib.Path
        Output directory where figures and the summary CSV will be saved.

    Returns
    -------
    None
        Side effects only (figure generation, CSV writing, and console prints). If no
        generalization rows are available, the function prints a message and returns early.

    Notes
    -----
    - Generalization rows are selected as ``df[df["split"] == "generalization"]``.
    - Model ordering is enforced by sorting on ``"model"`` prior to plotting.
    - One figure is saved per metric using the filename pattern::

          model_comparison_boxplot_distribution_<metric>.png

    - A summary CSV is saved as::

          model_comparison_boxplot_distribution_summary.csv

      The summary is built as ``df.groupby("model")[metrics_list].describe().T`` and then
      columns are sorted alphabetically by model name.

    Examples
    --------
    >>> import pandas as pd
    >>> from pathlib import Path
    >>> df = pd.DataFrame(
    ...     {
    ...         "model": ["A", "A", "B", "B"],
    ...         "split": ["generalization"] * 4,
    ...         "roc_auc": [0.91, 0.89, 0.93, 0.92],
    ...         "f1": [0.70, 0.68, 0.72, 0.71],
    ...     }
    ... )
    >>> compare_models_distribution(df, ["roc_auc", "f1"], Path("."))
    >>> True
    True
    """

    # 1. SORTING LOGIC: Alphabetical
    df.sort_values(by=["model"], inplace=True)

    # 2. Loop through each metric and create a SEPARATE plot
    for metric in metrics_list:
        plt.figure(figsize=(12, 10))
        sns.set_style("whitegrid")

        # B. Plot Boxplot (Summary)
        ax = sns.boxplot(
            data=df,
            x="model",
            hue="model",
            y=metric,
            palette="Spectral",
            width=0.5,
            fliersize=0,            # Hide outliers in boxplot
            linewidth=1.5,
            legend=False
        )

        # C. Plot Strip Plot (Raw Data)
        sns.stripplot(
            data=df,
            x="model",
            y=metric,
            color="#333333",
            alpha=0.35,
            size=3,
            jitter=0.25
        )

        # Styling
        plt.ylim([0, 1.05])
        plt.yticks(ticks=np.arange(0, 1.1, 0.1))
        plt.title(f"Model Comparison - Generalization Split: {metric.upper()}", fontweight="bold", pad=15)
        plt.xlabel("Model", fontweight="bold")
        plt.ylabel(f"{metric.upper()} Score", fontweight="bold")
        plt.xticks(rotation=90)
        plt.grid(True, axis="y", linestyle="--", alpha=0.5)

        # 3. Save Individual Figure
        fig_filename = f"model_comparison_distribution_boxplot_{metric}.png"
        plt.tight_layout()
        plt.savefig(save_path / fig_filename, dpi=300, bbox_inches="tight")
        plt.close()

        print(f"Saved comparison plot for {metric} -> {fig_filename}")

    # 4. Save Global Summary Table
    # We also sort the summary table index alphabetically for consistency
    summary = df.groupby("model")[metrics_list].describe().T
    summary = summary.sort_index(axis=1)
    summary.to_csv(save_path / "model_comparison_distribution_boxplot_summary.csv")

    print(f"\nAll plots and summary table saved to: {save_path}")

In [10]:
compare_models_distribution(generalization_df, metrics_to_analyze, FIGURES_MULTIPLE_MODELS_EVALUATION_DIR)

Saved comparison plot for balanced_accuracy -> model_comparison_distribution_boxplot_balanced_accuracy.png
Saved comparison plot for mcc -> model_comparison_distribution_boxplot_mcc.png
Saved comparison plot for average_precision -> model_comparison_distribution_boxplot_average_precision.png
Saved comparison plot for f1 -> model_comparison_distribution_boxplot_f1.png

All plots and summary table saved to: /home/leonardosaccotelli/Desktop/UNIVERSITA/MACHINE-LEARNING/Dynamic-Ensemble-Learning-for-Credit-Card-Fraud-Detection/reports/figures/EV_models_comparison_evaluation/CostSensitiveLearning___RandomizedSearchCV__niter_30__cv_5/multiple_models_evaluation


## Model Comparison (Mean Performance & Stability)

### Objective
To summarize the central tendency (Mean) and reliability (Standard Deviation) of each model on the Generalization split.

### Methodology
We generate **Bar Charts with Error Bars** for each metric:
* **Calculation:** We compute the **Mean** and **Standard Deviation** (SD) across all 100 data partitions (pooled 10 Iterations × 10 Folds).
    * **Bar Height:** Represents the Mean Score.
    * **Error Bar:** Represents ±1 Standard Deviation.
* **Sorting:** Models are sorted **Alphabetically by Name** for consistency with previous plots.
* **Scale:** Fixed to $0.0 - 1.0$ (or slightly higher for error bars) to ensure comparability.

### Interpretation Guide
* **Higher Bar:** Better average performance.
* **Shorter Error Bar:** More stable/reliable model.
* **Overlapping Error Bars:** If the error bars of two models overlap significantly, their performance difference may not be statistically significant (requires further testing).

In [11]:
def compare_models_barplot(df, metrics_list, save_path):
    """
    Compare models on the generalization split using per-metric mean barplots with standard-deviation error bars.

    This function filters the input results to the ``"generalization"`` split, sorts models
    alphabetically by the ``"model"`` column, and generates one bar plot per metric in
    ``metrics_list``. Each figure reports:
    - the mean metric value per model over all available generalization rows, and
    - the corresponding standard deviation as an error bar (via ``errorbar="sd"``).

    For each bar, the mean value is also annotated as a rotated text label.
    In addition to per-metric images, a summary CSV is saved with mean and standard
    deviation per model for all requested metrics.

    Parameters
    ----------
    df : pandas.DataFrame
        Input DataFrame containing evaluation results. Must include:
        - a ``"split"`` column containing the value ``"generalization"``,
        - a ``"model"`` column identifying the model name, and
        - numeric columns for each metric listed in ``metrics_list``.
    metrics_list : Sequence[str]
        Metric column names to plot and summarize. Each entry must exist in ``df`` and be
        numeric. The plotting routine assumes metric values are bounded in ``[0, 1]`` due
        to fixed y-axis limits (``0`` to ``1.05``).
    save_path : pathlib.Path
        Output directory where figures and the summary CSV are written. The function assumes
        the directory exists and is writable.

    Returns
    -------
    None
        Side effects only (plot generation, CSV writing, and console output). If no
        generalization rows are found, the function prints a message and returns early.

    Notes
    -----
    - The function expects to operate on generalization results; it filters data as
      ``df[df["split"] == "generalization"]``.
    - Model ordering is enforced by sorting on ``"model"`` and then extracting
      ``unique_models = df["model"].unique()`` for explicit bar ordering.
    - One figure is saved per metric with filename::

          model_comparison_barplot__mean_std_<metric>.png

    - A summary table is saved as::

          model_comparison_mean_std_summary.csv

      The summary is computed via ``df.groupby("model")[metrics_list].agg(["mean", "std"])``.

    Examples
    --------
    >>> import pandas as pd
    >>> from pathlib import Path
    >>> df = pd.DataFrame(
    ...     {
    ...         "model": ["A", "A", "B", "B"],
    ...         "split": ["generalization"] * 4,
    ...         "roc_auc": [0.91, 0.89, 0.93, 0.92],
    ...         "f1": [0.70, 0.68, 0.72, 0.71],
    ...     }
    ... )
    >>> compare_models_barplot(df, ["roc_auc", "f1"], Path("."))
    >>> True
    True
    """

    # 1. SORTING LOGIC: Alphabetical (Applied Once)
    # This ensures all subsequent plotting and grouping operations follow this order
    df.sort_values(by=["model"], inplace=True)
    unique_models = df["model"].unique()

    # 2. Loop through each metric
    for metric in metrics_list:
        plt.figure(figsize=(12, 10))
        sns.set_style("whitegrid")

        # B. Plot Bar Chart (Mean + Std Dev)
        # We don't need to pass 'order' explicitly because gen_data is already sorted
        ax = sns.barplot(
            data=df,
            x="model",
            y=metric,
            hue="model",
            palette="Spectral",
            errorbar="sd",             # Draw Error bar = Standard Deviation over the 100 points
            order=unique_models,       # Explicit ordering
            capsize=0.1,
            edgecolor="black",
            linewidth=1.2,
            alpha=0.85,
            legend=False
        )

        # C. Add Text Labels on Bars
        # We calculate means using the already-sorted dataframe
        # Note: groupby() sorts keys by default, but since we want to be safe,
        # we rely on the sorted dataframe's unique values for the loop order.
        means = df.groupby("model")[metric].mean()
        unique_models = df["model"].unique() # This preserves the sorted order from step A

        for i, model in enumerate(unique_models):
            score = means[model]
            ax.text(i, 0.05, f"{score:.3f}",
                    color="black", ha="center", va="bottom",
                    fontweight="bold", fontsize=12, rotation=90)

        # Styling
        plt.ylim([0, 1.05])
        plt.yticks(ticks=np.arange(0, 1.1, 0.1))

        plt.title(f"Model Mean Performance & Stability: {metric.upper()}", fontweight="bold", pad=20)
        plt.xlabel("Model", fontweight="bold")
        plt.ylabel(f"Mean {metric.upper()} (+/- Std Dev)", fontweight="bold")
        plt.xticks(rotation=90)
        plt.grid(True, axis='y', linestyle='--', alpha=0.5)

        # 3. Save Individual Figure
        fig_filename = f"model_comparison_mean_std_barplot_{metric}.png"
        plt.tight_layout()
        plt.savefig(save_path / fig_filename, dpi=300, bbox_inches="tight")
        plt.close()

        print(f"Saved barplot for {metric} -> {fig_filename}")

    # 4. Save Summary Table (Mean ± Std)
    summary = df.groupby("model")[metrics_list].agg(["mean", "std"])
    summary.to_csv(save_path / "model_comparison_mean_std_summary.csv")

    print(f"\nAll bar plots and summary table saved to: {save_path}")

In [12]:
compare_models_barplot(generalization_df, metrics_to_analyze, FIGURES_MULTIPLE_MODELS_EVALUATION_DIR)

Saved barplot for balanced_accuracy -> model_comparison_mean_std_barplot_balanced_accuracy.png
Saved barplot for mcc -> model_comparison_mean_std_barplot_mcc.png
Saved barplot for average_precision -> model_comparison_mean_std_barplot_average_precision.png
Saved barplot for f1 -> model_comparison_mean_std_barplot_f1.png

All bar plots and summary table saved to: /home/leonardosaccotelli/Desktop/UNIVERSITA/MACHINE-LEARNING/Dynamic-Ensemble-Learning-for-Credit-Card-Fraud-Detection/reports/figures/EV_models_comparison_evaluation/CostSensitiveLearning___RandomizedSearchCV__niter_30__cv_5/multiple_models_evaluation


In [13]:
def plot_significance_heatmap(df_comparisons, metric, save_path):
    """
    Plot a lower-triangular pairwise significance heatmap for a given metric.

    This function filters a pairwise-comparisons DataFrame to the requested metric,
    pivots the filtered rows into a p-value matrix (Model A × Model B), fills missing
    pairs with ``1.0`` (treated as non-significant), and renders a significance heatmap
    via :func:`scikit_posthocs.sign_plot`. The resulting figure is saved under
    ``save_path`` with a filename that includes the metric name.

    Parameters
    ----------
    df_comparisons : pandas.DataFrame
        Pairwise statistical comparison results. The DataFrame must include at least
        the following columns:

        - ``"Metric"``: metric identifier used to select rows for plotting.
        - ``"Model A"``: row model label in the p-value matrix.
        - ``"Model B"``: column model label in the p-value matrix.
        - ``"p-value"``: p-value associated with the (Model A, Model B) comparison.
    metric : str
        Metric name to plot. Must match one of the values in ``df_comparisons["Metric"]``.
    save_path : pathlib.Path
        Output directory where the heatmap image will be written.

    Returns
    -------
    None
        Side effects only (figure generation and file output).

    Notes
    -----
    - The p-value matrix is created as::

          df_metric = df_comparisons[df_comparisons["Metric"] == metric].copy()
          p_matrix = df_metric.pivot(index="Model A", columns="Model B", values="p-value")
          p_matrix = p_matrix.fillna(1.0)

      Missing comparisons are filled with ``1.0`` so they appear as non-significant.
    - The plot is produced using a user-defined categorical colormap list and
      additional styling options forwarded to :func:`scikit_posthocs.sign_plot`.
    - The figure title includes the metric name and indicates the corrected resampled
      t-test context.
    - The saved filename is::

          model_comparison_corrected_resampled_ttest_heatmap_<metric>.png

    Examples
    --------
    >>> import pandas as pd
    >>> from pathlib import Path
    >>> df_comp = pd.DataFrame(
    ...     {
    ...         "Metric": ["roc_auc", "roc_auc"],
    ...         "Model A": ["A", "A"],
    ...         "Model B": ["B", "C"],
    ...         "p-value": [0.03, 0.20],
    ...     }
    ... )
    >>> plot_significance_heatmap(df_comp, "roc_auc", Path("."))
    >>> True
    True
    """

    # 1. FILTER: Select only the rows for the current metric
    df_metric = df_comparisons[df_comparisons["Metric"] == metric].copy()

    # 2. Pivot to get the P-Value Matrix
    p_matrix = df_metric.pivot(index="Model_A", columns="Model_B", values="p-value")

    # 3. Fill NaNs with 1.0 (non-significant) initially to handle missing pairs
    p_matrix = p_matrix.fillna(1.0)

    # 4. Plot using scikit-posthocs
    plt.figure(figsize=(10, 8))

    # Define Colormap:
    # 1 (White), 0.05, 0.01, 0.001 (Blues/Reds)
    cmap = ["1", "#fb6a4a", "#08306b", "#4292c6", "#c6dbef"]

    heatmap_args = {
        "cmap": cmap,
        "linewidths": 0.5,  # Slightly thicker lines for clarity
        "linecolor": "0.9",  # Light gray grid lines
        "clip_on": False,
        "square": True,  # Force square cells
        # [x, y, width, height] in figure coordinate system
        # Moved x to 1.02 to place it outside the plot area
        "cbar_ax_bbox": [0.90, 0.35, 0.03, 0.3],
    }

    # Plot
    sp.sign_plot(p_matrix, **heatmap_args)

    plt.suptitle(
        f"Pairwise Significance: {metric}\n(Corrected Resampled t-test)", fontweight="bold"
    )

    # Adjust layout to accommodate the external legend
    plt.subplots_adjust(right=0.85)

    plt.savefig(
        save_path / f"model_comparison_corrected_resampled_ttest_heatmap_{metric}.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.close()

In [14]:
def compute_pairwise_ttests(df, metrics_list, n_train, n_test, save_path, alpha=0.05):
    """
    Computes pairwise tests, saves CSV, and generates visualizations.
    """

    df.sort_values(by=["model"], inplace=True)
    unique_models = df["model"].unique()

    comparison_records = []

    print(f"Processing {len(unique_models)} models and {len(metrics_list)} metrics...")

    # Compute Statistics
    for metric in metrics_list:
        for model_a in unique_models:
            for model_b in unique_models:
                if model_a == model_b: continue

                vec_a = df[df["model"] == model_a][metric].values
                vec_b = df[df["model"] == model_b][metric].values

                if len(vec_a) != len(vec_b): continue

                t_stat, p_val = corrected_resampled_ttest(vec_a, vec_b, n_train, n_test)
                is_significant = p_val < alpha

                if is_significant and t_stat > 0:
                    relationship = "Better" # A is better than B
                elif is_significant and t_stat < 0:
                    relationship = "Worse"  # A is worse than B
                else:
                    relationship = "No Diff"

                comparison_records.append({
                    "Metric": metric,
                    "Model_A": model_a,
                    "Model_B": model_b,
                    "t-stat": t_stat,
                    "p-value": p_val,
                    "is_significant": is_significant,
                    "result": relationship,
                })

    df_comparisons = pd.DataFrame(comparison_records)

    # Save CSV
    df_comparisons.to_csv(save_path / "model_comparison_corrected_resampled_ttest.csv", index=False)

    # Generate Visualizations
    for metric in metrics_list:
        print(f"Generating plots for {metric}...")

        # Scikit-Posthocs heatmap of significance
        plot_significance_heatmap(df_comparisons, metric, save_path)

    print("All visualizations generated.")
    return df_comparisons

In [15]:
_ = compute_pairwise_ttests(generalization_df, metrics_to_analyze, 0.9, 0.1, FIGURES_MULTIPLE_MODELS_EVALUATION_DIR)

Processing 23 models and 4 metrics...
Generating plots for balanced_accuracy...
Generating plots for mcc...
Generating plots for average_precision...
Generating plots for f1...
All visualizations generated.
